# Humano no Loop: Portões Pré-Ação, Classificação de Risco e Registro de Auditoria

O README desta lição introduz Humano no Loop com um pequeno trecho que pergunta ao usuário `APROVAR` ou `REJEITAR` depois que o agente já produziu uma resposta. Esse padrão é um bom ponto de partida, mas implementações de HITL em produção normalmente precisam de três peças adicionais:

1. Um **portão pré-ação** que roda **antes** do agente executar uma etapa arriscada, para que custo, irreversibilidade e latência fiquem sob controle.
2. **Classificação de risco**, para que ações de baixo risco sejam executadas automaticamente, ações de risco médio sejam aprovadas em lote, e apenas ações de alto risco dependam de um humano.
3. Um **registro de auditoria mais um ciclo de revisão**, para que toda decisão do portão seja registrada como JSONL, e uma rejeição reprovoque o agente com uma razão estruturada em vez de apenas imprimir `Revisando...`.

Este notebook constrói cada um desses em cima dos mesmos primitivos que `06-system-message-framework.ipynb`. Ele roda de ponta a ponta em `DEMO_MODE = True` (sem necessidade de entrada interativa) ou com prompts reais `input()` quando `DEMO_MODE = False`. Nota: no DEMO_MODE a tentativa de repetição do terceiro objetivo é roteirizada para que a mecânica do loop seja visível de ponta a ponta. Real reclassificação orientada por revisão requer `DEMO_MODE = False` e um operador.

**Fora do escopo (abordado em outras lições):** autenticação e controle de acesso (ameaça 2 do README da Lição 06), middleware de chamada de ferramentas (exploração profunda do MAF na Lição 14), padrões de debate multiagente.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## Padrão 1: Porta pré-ação

O trecho HITL do README chama o agente primeiro, depois pede ao usuário que aprove a saída. Esse é um fluxo de **pós-ação**. O agente já executou, então o custo da chamada LLM já foi pago, e qualquer efeito colateral (email enviado, linha de banco de dados escrita, comentário postado) já ocorreu.

Um fluxo de **pré-ação** insere a porta antes que o agente execute a etapa arriscada. O agente propõe a ação, a porta decide se executa, e somente com a aprovação o efeito colateral ocorre.

| Aspecto | Aprovação pós-ação (trecho do README) | Porta pré-ação (este notebook) |
|---|---|---|
| Quando a aprovação ocorre? | Depois que o agente produziu a saída | Antes de qualquer efeito colateral ser executado |
| Custo LLM em caso de rejeição | Já pago | Pago apenas pela proposta, não pela ação |
| Efeitos colaterais irreversíveis | Possível (a ação já aconteceu) | Prevenido |
| Clareza na auditoria | Aprovação é uma declaração print | Aprovação é um registro JSON com timestamp, ação, motivo |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Padrão 2: Classificação de risco

Nem toda ação precisa de aprovação humana. Uma consulta somente leitura a uma API pública tem riscos diferentes de enviar um e-mail para um cliente. Tratar ambos da mesma forma desperdiça a atenção do operador e desacelera o agente.

Um modelo simples de 3 níveis:

| Nível | Exemplos | Fluxo de aprovação |
|---|---|---|
| `baixo` (somente leitura) | Pesquisar uma base de conhecimento, consultar opções de voo, buscar uma página web pública | Execução automática, registrada para auditoria |
| `médio` (mutação barata) | Armazenar em cache um resultado, incrementar um contador, agendar um lembrete | Execução automática, mas com revisão diária em lote |
| `alto` (externo ou irreversível) | Enviar um e-mail, cobrar um cartão, postar em um canal público | Bloqueado para aprovação humana |

Esta é uma classificação. Sistemas em produção geralmente usam níveis mais granulares (por exemplo, níveis de permissão do AWS IAM, níveis de acesso baseados em funções). A versão de 3 níveis abaixo é a menor versão útil para um agente que mistura ações somente leitura e que causam efeitos colaterais.

O classificador abaixo usa heurísticas de palavras-chave para que a demonstração permaneça determinística e barata. Em um sistema de produção, você substituiria isso por um classificador treinado ou um mecanismo de políticas.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Padrão 3: Log de auditoria e loop de revisão

Um `print("Resposta aprovada.")` não é um log de auditoria. Para confiança, toda decisão do portão deve ser registrada como um evento estruturado que você pode consultar depois, reproduzir ou anexar a uma revisão de incidente.

Dois pontos:

1. **JSONL somente anexado.** Uma linha por decisão, com timestamp, ação, nível, decisão, motivo. Fácil de buscar, fácil de enviar para um repositório de logs real depois.
2. **Loop de revisão em rejeição.** Quando o portão retorna `deny`, o agente se re-invoca com o motivo da rejeição no contexto, para que a próxima proposta evite o problema.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Recursos adicionais

Vários outros projetos públicos implementam variações desses padrões HITL. Compare abordagens para encontrar o que se encaixa na sua pilha:

- **LangChain** wrappers de ferramentas human-in-the-loop ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): wrappers de ferramentas plug-and-play que pausam a execução para entrada humana.
- **AutoGen** `UserProxyAgent` ([docs v0.2](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ reestruturado): usa um papel de agente especificamente para representar o humano em conversas multi-agentes.
- **Microsoft Agent Framework (MAF)** middleware de invocação de função ([docs](https://learn.microsoft.com/agent-framework/)): middleware que roda em torno de toda chamada de ferramenta/função, adequado para lógica de controle e fluxos de aprovação.

Cada projeto lida com os três sub-padrões de forma diferente: LangChain os envolve como ferramentas, AutoGen usa um papel de agente, e Microsoft Agent Framework usa middleware de invocação de função. Leia uma ou duas implementações completas antes de escolher um design para seu próprio agente.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido usando o serviço de tradução por IA [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, por favor, esteja ciente de que traduções automatizadas podem conter erros ou imprecisões. O documento original em seu idioma nativo deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
